# Modelos latentes: V-JEPA 2, V-JEPA 2-AC y LeWM
Tres modelos que **no generan video**: aprenden un espacio de embeddings y predicen en él. No tiene sentido medirlos con FVD, así que se los compara con
otras cosas: error de predicción contra *copiar el último latente*, probes de pose contra un encoder sin entrenar y contra píxeles, sorpresa ante un
corte del video y recuperación de acciones con CEM. Todo lo que sigue sale de `results/latent_<corrida>.json`; sin él, *pendiente*.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# la raíz del repo: sube desde donde esté abierto el notebook hasta encontrar src/common.py
RAIZ = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "src" / "common.py").exists()), None)
if RAIZ is None:
  raise SystemExit("abrí el notebook dentro del repo (en Colab, corré antes la primera celda de notebooks/colab.ipynb)")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src import nbui
from src.toolkit import Toolkit

# scripts/run_notebooks.py pone ML2_NB_LIMITE para correr todo chico; a mano queda en None (sin recorte)
LIMITE = int(os.environ.get("ML2_NB_LIMITE", "0")) or None

## 1. Qué es cada uno
| | V-JEPA 2 | V-JEPA 2-AC | LeWM |
|---|---|---|---|
| Qué aprende | un encoder ViT que predice los tubelets tapados | un predictor de latentes condicionado por acciones, sobre un encoder congelado | un encoder + predictor de un embedding por frame |
| Objetivo | L1 contra un target EMA con stop-grad | L1 con teacher forcing y rollout de 2 pasos | error de predicción + SIGReg (evita el colapso) |
| Acciones | no | sí (pose y diferencia de pose) | sí (5 poses apiladas) |
| Planifica | no | CEM (800 muestras, top-10) | CEM |
| CLI | `vjepa2` | `vjepa2_ac` (necesita un `vjepa2`) | `lewm` |

## 2. Receta y fidelidad de cada uno

In [ ]:
from src import recipes
nbui.tabla_fidelidad(["vjepa2", "vjepa2_ac", "lewm"]);

## 3. Máscaras de V-JEPA 2
El predictor solo ve los tokens visibles de un clip tapado en ~90 % con bloques 3D; tiene que adivinar el resto en el espacio latente. No dependen de los pesos.

In [ ]:
nbui.mascaras_vjepa2(semilla=0);

## 4. Comparación
Una fila por modelo, con sus referencias al lado.

In [ ]:
nbui.tabla_latente();

### Error por horizonte
Contra copiar el último latente y contra la acción nula (si la acción importa, la nula es peor que la real).

In [ ]:
for run in ("lewm", "vjepa2_ac"):
  nbui.horizonte_latente(run)

### Probes de pose
R² de velocidad y giro sobre el encoder entrenado, el encoder sin entrenar y los píxeles.

In [ ]:
for run in ("lewm", "vjepa2_s"):
  nbui.probes_fig(run)

### Sorpresa
El error de predicción salta cuando el video se corta (teletransporte o salto de 2 s) si el modelo entiende la escena.

In [ ]:
for run in ("lewm", "vjepa2_ac"):
  nbui.sorpresa_fig(run)

### Plan con CEM
Error de la pose planeada contra repetir la última acción y contra la media de train.

In [ ]:
for run in ("lewm", "vjepa2_ac"):
  nbui.cem_tabla(run)

## 5. Correrlos
Cada uno con la misma API de siempre (`DRY_RUN = True` imprime los comandos; `False` los corre en segundo plano y queda en la bitácora). Los notebooks por modelo (`notebooks/modelos/lewm.ipynb`, `vjepa2.ipynb`, `vjepa2_ac.ipynb`) tienen todas las acciones.

In [ ]:
DRY_RUN = True
for modelo, opts in (("vjepa2", dict(tamano="s")), ("vjepa2_ac", dict(encoder="vjepa2_s")), ("lewm", {})):
  print("##", modelo)
  nbui.lanzar(modelo, "entrenar", dry_run=DRY_RUN, **opts)
  nbui.lanzar(modelo, "evaluar", dry_run=DRY_RUN, limite=LIMITE)